<a href="https://colab.research.google.com/github/Jupeid/interactivebook/blob/main/TesteLivroInt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
class Personagem:
  def __init__(self,nome):
    self.nome = nome
    self.nivel = 0
    self.experiencia = 0
    self.exp_para_proximo_nivel = 10
    self.pontos_disponiveis = 0
    self.pode_distribuir_pontos = False
# ------ATRIBUTOS BASE------
    self.forca = 0
    self.destreza = 0
    self.inteligencia = 1
    self.sorte = 1
    self.magia = 0
    self.vitalidade = 0
# ------STATUS DERIVADOS------
    self.hp_max = 20 + (self.vitalidade * 5)
    self.hp_atual = self.hp_max
    self.mp_max = self.magia * 5
    self.mp_atual = self.mp_max
# -----INVENTARIO------
    self.inventario = []
# ------METODO DE TRAVA----
  def permitir_distribuicao(self, permitir:bool):
    self.pode_distribuir_pontos = permitir
# ------ATT DE STATUS------
  def recalcular_status_maximos(self):
    self.hp_max = 20 + (self.vitalidade * 5)
    self.mp_max = self.magia * 5
# ------TESTE ATRIBUTOS-----
  def tem_atributo(self, nome_atributo, valor_minimo):
    valor_atual = getattr(self, nome_atributo.lower(), 0)
    return valor_atual >= valor_minimo

  def tem_mp(self, custo_mp):
    return self.mp_atual >= custo_mp
# -----ALTERA ATRIBUTOS-----
  def consumir_mp(self, quantidade):
    if self.tem_mp(quantidade):
      self.mp_atual -= quantidade
      print(f"\n [MP] Você gastou {quantidade} de MP. MP Atual: {self.mp_atual}/{self.mp_max}")
      return True
    return False

  def receber_dano(self, dano):
    self.hp_atual = max(0, self.hp_atual - dano)
    print(f"\n [DANO] Você sofreu {dano} de dano! HP: {self.hp_atual}/{self.hp_max}")

  def descansar(self):
    self.hp_atual = self.hp_max
    self.mp_atual = self.mp_max
    print("\n [DESCANSO] Seu HP e MP forma totalmente restaurados!")

# -----SISTEMA DE NIVEL E EXPERIENCIA-----
  def ganhar_xp(self, quantidade_xp):
    self.experiencia += quantidade_xp
    print(f"\n [XP] Você ganhou {quantidade_xp} de XP! ({self.experiencia}/{self.exp_para_proximo_nivel})")

    while self.experiencia >= self.exp_para_proximo_nivel:
        self.experiencia -= self.exp_para_proximo_nivel
        self.nivel += 1
        self.pontos_disponiveis += 1
        self.exp_para_proximo_nivel = int(self.exp_para_proximo_nivel * 1.5)
        print(f"\n NIVEL UP! Você subiu para o Nível {self.nivel}!")
        print(f"Você tem {self.pontos_disponiveis} ponto(s) de status para distribuir.")

  def distribuir_pontos(self, atributo):
    if not self.pode_distribuir_pontos:
      print("\n [!] Você só pode distribuir pontos em áreas de descanso"
      " ou trocas de capítulo!")
      return False

    if self.pontos_disponiveis <= 0:
      print("Você não possui pontos de status disponíveis!")
      return False

    atributo = atributo.lower()
    atributos_validos = [
        "forca",
        "destreza",
        "inteligencia",
        "sorte",
        "magia",
        "vitalidade",
    ]
    if atributo in atributos_validos and hasattr(self, atributo):
      valor_antigo = getattr(self, atributo)
      setattr(self, atributo, valor_antigo + 1)
      self.pontos_disponiveis -= 1

      self.recalcular_status_maximos()

      if atributo == "vitalidade":
        self.hp_atual += 5
      elif atributo == "magia":
        self.mp_atual += 5

      print(f"\n [STATUS] {atributo.capitalize()} aumentado para {getattr(self, atributo)}!")
      return True
    else:
      print(f"[!] Atributo inválido!")
      return False

In [ ]:
def testar_requisitos(jogador, lista_requisitos):
  for req in lista_requisitos:
    tipo = req["tipo"]

    if tipo == "atributo":
      if not jogador.tem_atributo(req["nome"], req["valor"]):
        return False
    elif tipo == "magia":
      tem_nivel = jogador.tem_atributo(req["nome"], req["valor"])
      tem_mp = jogador.tem_mp(req["custo_mp"])
      if not (tem_nivel and tem_mp):
        return False
    elif tipo == "item":
      if req["nome"] not in [i.nome for i in jogador.inventario]:
        return False
  return True

In [ ]:
def processar_escolha(jogador, opcao_escolhida):
  if "proxima_cena" in opcao_escolhida:
    return opcao_escolhida["proxima_cena"]

  modos = opcao_escolhida.get("modos", [])
  modo_bem_sucedido = None

  for modo in modos:
    if testar_requisitos(jogador, modo["requisitos"]):
      modo_bem_sucedido = modo
      break
  if modo_bem_sucedido:
    for req in modo_bem_sucedido["requisitos"]:
      if req["tipo"] == "magia":
        jogador.consumir_mp(req["custo_mp"])

    print("\n" + "=" * 40)
    print(modo_bem_sucedido["narrativa"])
    print("=" * 40 + "\n")

    return modo_bem_sucedido["proxima_cena"]

  else:
    print("\n[!] Escolha inválida!" )
    return None

In [ ]:
def exibir_hud(jogador):
  print("=" * 50)
  print(f"Nome: {jogador.nome} | Nível: {jogador.nivel}")
  print(f"HP: {jogador.hp_atual}/{jogador.hp_max} | MP: {jogador.mp_atual}/{jogador.mp_max}")
  print(f"Força: {jogador.forca} | Destreza: {jogador.destreza}")
  print(f"Inteligência: {jogador.inteligencia} | Sorte: {jogador.sorte}")
  print(f"Magia: {jogador.magia} | Vitalidade: {jogador.vitalidade}")
  print("=" * 50)

In [ ]:
def rodar_cena(jogador, cena):
  permite_descanso = cena.get("permite_descanso", False)
  jogador.permitir_distribuicao(permite_descanso)
  if permite_descanso:
    jogador.descansar()

  while True:
    exibir_hud(jogador)

    titulo = cena.get("titulo")

    if titulo:
      print("\n" + "=" * 40)
      print(cena["titulo"])
      print("=" * 40)
    else:
      print("\n")

    print(cena["narrativa"])
    print("\nOpções:")
    for letra, dados_opcao in cena["opcoes"].items():
      print(f"[{letra}] {dados_opcao['texto']}")

    entrada = input("\nEscolha uma opção: ").strip().upper()
    if entrada not in cena["opcoes"]:
      print("\n[!] Escolha inválida!")
      continue

    opcoes_selecionadas = cena["opcoes"][entrada]
    proxima_cena = processar_escolha(jogador, opcoes_selecionadas)

    if proxima_cena is not None:
      return proxima_cena

    input("Pressione Enter para tentar novamente...")

In [ ]:
cenas = {
    "prologo": {
    "titulo": "PRÓLOGO - O JOGO DIVINO",
    "narrativa": """
Em um salão celestial banhado por uma luz dourada, oito entidades se reúnem
ao redor de uma imensa mesa circular ornada em ouro. Ao fundo da sala, sobre um
pedestal elevado, repousa um trono majestoso, porém completamente vazio.
Em ambos os lados desse assento principal, dois tronos menores completam a
arquitetura divina.
No trono da direita, descansa uma figura de pele azulada, envolta em robes
negros trespassados por linhas douradas que se movem como areia viva. O capuz
cobre-lhe a cabeça e sua feição permanece imutável, enquanto seus olhos azuis
brilhantes observam em silêncio o debate dos deuses abaixo. Ao seu lado,
no trono da esquerda, dividem o espaço duas figuras: um homem adulto,
aparentando trinta anos, vestindo robes negros encapuzados e portando olhos e
cabelos tão escuros quanto o vazio; em seu colo, descansa uma garotinha de
aproximadamente oito anos, vestida em túnicas alvas, com olhos e cabelos
reluzentes como ouro. Descontraída e alheia à tensão do recinto, a criança
brinca com uma pequena árvore que brota da palma de sua mão, um verdadeiro
ecossistema vivo onde minúsculas fadas e animais habitam entre os galhos.

Iniciando o debate, uma figura de grande porte e postura imponente bate a mão
sobre a mesa com firmeza calculada:

— Já se passaram eras desde que o Trono Supremo esfriou. Continuar fingindo que
a ausência do nosso Criador é apenas algo temporário é uma irresponsabilidade
com a própria existência! O cosmos precisa de um pilar. Precisamos de alguém com
firmeza e liderança para manter o comando antes que o caos se alastre.

Observando a reação do guerreiro, uma jovem dama diz em tom melodioso e
levemente melancólico:

— Entendo sua preocupação, meu caro... Mas me corta o coração ver essa sede por
substituição. Não deveríamos, em primeiro lugar, canalizar nossos esforços para
encontrar Nosso Pai? Onde quer que ele esteja, meu ser clama por sua presença.
Assumir aquele assento tão levianamente me parece quase uma profanação...

Como se esperasse a oportunidade perfeita para intervir, uma terceira voz ressoa
pelo salão. O orador sorri sob a sombra de seu capuz, ajustando os anéis
dourados que reluzem no escuro de suas mangas:

— Uma busca exige recursos e estabilidade, querida. Se a preocupação da mesa é o
atrito sobre quem deve sentar-se ali, proponho uma solução: só é digno de ocupar
o trono aquele capaz de observar a existência de maneira imparcial.
Eu, por exemplo, posso garantir que os interesses de todas as facções sejam
mantidos em perfeito equilíbrio até que nosso Patrono retorne. Assim, evitamos
uma disputa desnecessária enquanto garantimos que o cosmos não se desestabilize
por mais tempo.

Com a voz ligeiramente trêmula, apertando as próprias mãos ocultas sob as vestes,
outra jovem protesta:

— "Dignidade"? "Imparcialidade"? Vocês falam como se a ordem divina fosse algo
que pudéssemos controlar! Há algo errado nas correntes arcanas do cosmos...
A essência Dele não está apenas ausente, ela parece... fragmentada. Sentar
naquele trono agora pode desencadear uma reação mágica irreversível. Somente o
nosso Criador teria capacidade de suportar tal fado. Quem somos nós para nos
acharmos dignos de ser seus substitutos?

Com a voz calma, porém cortante e transbordando desconfiança em relação às
intenções do Comércio e da Guerra, o senhor de feições mais antigas da mesa
declara:

— Concordo com a Magia. Sugerir um ocupante temporário nada mais é do que uma
tentativa falha de controlar algo que nem mesmo nós conseguimos compreender.
O Trono de Ouro pertence àquele que nos moldou. Nenhum de nós possui a
autoridade necessária para ocupá-lo, nem por um único segundo.

Enquanto a discussão esquenta, no canto da mesa, uma jovem pequena ignora
completamente a discórdia. Ela se levanta devagar, ajeita seu vestido verde e
aproxima-se do trono de Vida e Morte. Com toque suave, roça a pequena árvore na
palma da garotinha; um brilho esmeralda emerge imediatamente, fazendo brotar
frutos dourados que alimentam as fadas e animais. Do outro lado da mesa,
o Entretenimento apenas solta uma risada abafada enquanto joga cartas contra
três ilusões perfeitas de si mesmo, fazendo pilhas de baralhos flutuarem no ar.

Interrompendo a discussão com um tom ríspido, uma figura feminina de ar maduro
levanta-se subitamente:

— Silêncio! Vocês discutem como reis mimados enquanto o mundo mortal treme
lá embaixo. Enquanto vocês debatem interminavelmente, eu consegui identificar
traços sutis de essência divina vazando diretamente para o plano terrestre.
Se Nosso Patrono desceu até os mortais, precisamos nos mover o mais rápido
possível e trazê-lo de volta! Se me permitirem, descerei para encontrar um
rastro e questionar o motivo de sua ausência.

As deidades na mesa se calam e se entreolham. A ideia de uma deidade interferir
e caminhar no plano mortal traz um peso perigoso.

Do trono menor à direita, a figura de pele azulada ergue-se devagar.
O som de areia fluindo sob seus robes negros ecoa pelo salão, e o simples ato de
se levantar faz a mesa circular emudecer instantaneamente. Sua voz ressoa não
nos ouvidos, mas diretamente na mente de todos:

— Sua presença no plano mortal destruiria o equilíbrio dos reinos, Caça. Se uma
deidade pisar na terra em sua forma verdadeira, as leis da existência ruirão.

A figura estende as mãos, e oito orbes de luz pulsante e enigmática emergem de
suas palmas.

— Porém, devo concordar com um ponto... O pensamento de que o trono desocupado
está afetando o equilíbrio do cosmos não está totalmente errado. Contudo, no
caminho de conflito que vocês estão seguindo, não enxergo outro desfecho que não
seja a ruína. Sendo assim, sugiro um meio para resolver a questão do ocupante do
trono: cada um de vocês escolherá um mortal. Um campeão para carregar uma
centelha de nosso poder, moldada em um artefato único.

Os orbes flutuam e param diante de cada um dos oito deuses.

— Depositem nesses orbes parte de suas essências divinas para que carreguem suas
vontades. Quando o Criador desapareceu para conter um desequilíbrio maior no
cosmos, moldei o destino para que nascessem oito receptáculos capazes de
suportar e carregar suas essências. Seus campeões lutarão, investigarão e
ascenderão no plano mortal. Aquele cujo campeão prevalecer... herdará
o Trono de Ouro.""",
    "opcoes": {
        "A": {
        "texto": "Continuar para o Capítulo 1",
        "proxima_cena": "capitulo_1",

          }

        }
    },

"capitulo_1": {
      "titulo": "Capítulo 1 – O Primeiro Encontro",
      "narrativa": """
Na manhã de segunda-feira, desperto na minha cama com uma enxaqueca terrível que
começou na noite anterior. Levanto-me preguiçosamente para me arrumar, lanço um
olhar ao relógio na cabeceira e vejo que ainda restam duas horas para a
cerimônia de abertura.

Após um longo banho quente, visto o uniforme da instituição: um terno
azul-marinho de corte elegante, ajustado sobre uma camisa social branca. Preparo
um café da manhã simples — pão, ovos e café preto coado — e saio de casa a
caminho do ponto de ônibus.

Hoje é o meu primeiro dia, e sinto uma mistura de empolgação e ansiedade ao
pensar no que me aguarda na maior e mais respeitada instituição do continente de
Aurelis: a Consortium Academia. Criada pelas três maiores potências para
garantir o recrutamento justo e manter o equilíbrio de poder entre as nações,
a academia é um verdadeiro campo de provações. Ali, jovens como eu tentam
demonstrar e desenvolver seus talentos para retornar à terra natal cobertos de
glória ou ser contratados por potências rivais em busca de ascensão.

Para alguém como eu, vindo do Reino de Drakkenheim, frequentar um lugar como
aquele beirava o impossível. No entanto, um ano atrás, minha sorte mudou
drasticamente.

Enquanto ajudava minha família no campo, notei uma movimentação estranha na mata
ao redor. Movido pela curiosidade, aproximei-me e encontrei uma jovem
desacordada, amarrada ao tronco de uma árvore por cordas grossas. Quem quer
que a tivesse deixado ali não pretendia permitir seu retorno: ao redor
do tronco, vários incensos queimavam, exalando um aroma adocicado que
estimulava os predadores locais. Eu conhecia bem aquele cheiro.
Era o mesmo artifício usado pelo meu pai para atrair as feras da floresta para
longe do vilarejo durante as criações de outbreaks — surtos de feras comuns na
nossa região quando a caça escasseava.

Três lobos selvagens cercavam a garota, encarando-a com olhos famintos. Mas,
no instante em que me preparava para agir, o tempo ao meu redor pareceu
desacelerar drasticamente. Em minha mente, uma voz feminina, alegre e jovial,
ressoou com clareza:

— Te encontrei!…

Logo em seguida, uma interface translúcida de tom dourado emergiu no ar,
exibindo uma mensagem cristalina diante dos meus olhos:

===========================================================
[Missão de Aura]
• Objetivo: Impeça que uma vida seja perdida. Salve a garota.
• Recompensa: Oportunidade Única de Evolução

[Nota: Missão Extra — Um Presente de Nox]
• Objetivo: Vidas estão prestes a encontrar seu fim, porém o
  destino pode ser alterado. Elimine dois lobos para manter o
  equilíbrio.
• Recompensa: Item Único
===========================================================

De volta à realidade, noto a garota despertando. Ela possui olhos brilhantes da
mesma cor de seus cabelos — um tom de lilás vivo e marcante —, emoldurando um
rosto de feições doces. Sem dúvidas, é uma das meninas mais belas que já vi e,
por razões desconhecidas, me transmite uma estranha sensação de familiaridade.
Ao perceber os lobos se aproximando, ela tenta se soltar violentamente, mas os
nós são firmes demais. Ela murmura palavras incompreensíveis enquanto sinto uma
ondulação visível de energia se acumular em suas mãos. (Ela é uma maga!), deduzo
imediatamente. Contudo, seu semblante antes calmo assume uma expressão
desesperada: a mana não se condensa rápido o suficiente para conjurar o feitiço
antes do bote dos predadores.

Ela olha ao redor e, finalmente, nota a minha presença.

— Me ajude, por favor! — clama ela, com a voz embargada pelo pânico.
— Minha família tem dinheiro! Posso conseguir o que quiser se me tirar daqui!

Despertando do transe momentâneo, encaro a urgência diante de mim.
      """,
      "opcoes": {
          "A": {
              "texto": """Tentar atrair a atenção dos lobos, criando uma brecha
para que a garota finalize a conjuração do feitiço.""",
              "modos": [
                  {
                      "requisitos": [{"tipo": "atributo", "nome": "inteligencia", "valor":1}],
                      "narrativa": "",
                      "proxima_cena": "capitulo_1_resgate",
                  }
              ],
            },
          "B": {
              "texto": """Colocar-me entre os lobos e a garota, destruindo os
incensos na esperança de dispersar a agressividade dos predadores.""",
              "modos": [
                  {
                      "requisitos": [],
                      "narrativa": "",
                      "proxima_cena": "capitulo_1_resgate",
                  }
              ],
            },
          "C": {
              "texto": """Ignorar o apelo da garota e a notificação do Sistema,
virando as costas e indo embora.""",
              "modos": [
                  {
                      "requisitos": [],
                      "narrativa": "",
                      "proxima_cena": "capitulo_1_abandono",
                  }
              ],
            },
      },

    }
  }


In [ ]:
protagonista = Personagem("Kael")
cena_atual_id = "prologo"

while cena_atual_id in cenas and cenas[cena_atual_id]["opcoes"]:
  cena_objeto = cenas[cena_atual_id]
  cena_atual_id = rodar_cena(protagonista, cena_objeto)

Nome: Kael | Nível: 0
HP: 20/20 | MP: 0/0
Força: 0 | Destreza: 0
Inteligência: 1 | Sorte: 1
Magia: 0 | Vitalidade: 0

PRÓLOGO - O JOGO DIVINO

Em um salão celestial banhado por uma luz dourada, oito entidades se reúnem
ao redor de uma imensa mesa circular ornada em ouro. Ao fundo da sala, sobre um
pedestal elevado, repousa um trono majestoso, porém completamente vazio.
Em ambos os lados desse assento principal, dois tronos menores completam a
arquitetura divina.
No trono da direita, descansa uma figura de pele azulada, envolta em robes
negros trespassados por linhas douradas que se movem como areia viva. O capuz
cobre-lhe a cabeça e sua feição permanece imutável, enquanto seus olhos azuis
brilhantes observam em silêncio o debate dos deuses abaixo. Ao seu lado,
no trono da esquerda, dividem o espaço duas figuras: um homem adulto,
aparentando trinta anos, vestindo robes negros encapuzados e portando olhos e
cabelos tão escuros quanto o vazio; em seu colo, descansa uma garotinha de
aproxi

KeyboardInterrupt: Interrupted by user